# Cohort: from 1,863 meniscal injuries to the final cohort

**Purpose.** This notebook builds the cohort of the project *What happens to professional football players after a serious meniscal injury in Europe's top five leagues?* It applies, step by step, the rules fixed in `01_data_audit.ipynb`, counts injuries and players after each step, and saves one row per player.

**Inputs.** The six tables of the audit (Transfermarkt, via Kaggle), loaded the same way.

**Checks.** After every step the numbers of injuries and players are compared with the numbers found in the audit; the notebook stops if they differ.

**Output.** The cohort file read by `03_data_analysis.ipynb`. The analysis notebook does not touch the raw tables.

## Step 0

### 0.1 Load the tables and 0.2 convert dates

In the CSV files dates are text. We convert them to dates and **count how many values failed to convert** (`errors="coerce"` silently turns them into missing values, so without this check the errors would disappear).

In [1]:
# 0.1 Setup and loading 
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

KAGGLE_A = Path("/kaggle/input/datasets/xfkzujqjvx97n/football-datasets")
KAGGLE_B = Path("/kaggle/input/datasets/davidcariboo/player-scores")

def csv_path(name):
    candidates = [KAGGLE_A / name / f"{name}.csv", KAGGLE_B / f"{name}.csv",
                  Path("data") / f"{name}.csv", Path("../data") / f"{name}.csv"]
    found = next((p for p in candidates if p.exists()), None)
    if found is None:
        raise FileNotFoundError(f"{name}.csv not found; looked in: {[str(p) for p in candidates]}")
    return found

player_injuries     = pd.read_csv(csv_path("player_injuries"))
player_performances = pd.read_csv(csv_path("player_performances"))
players             = pd.read_csv(csv_path("players"))
appearances         = pd.read_csv(csv_path("appearances"))
games               = pd.read_csv(csv_path("games"))
transfers           = pd.read_csv(csv_path("transfers"))

tables = {"player_injuries": player_injuries, "player_performances": player_performances,
          "players": players, "appearances": appearances, "games": games, "transfers": transfers}
for name, df in tables.items():
    print(f"{name:20s} rows: {len(df):>9,}   columns: {df.shape[1]}")

# 0.2 Dates: conversion and range 
date_cols = {"player_injuries": ["from_date", "end_date"], "appearances": ["date"], "games": ["date"],
             "transfers": ["transfer_date"], "players": ["date_of_birth"]}
rows = []
for name, cols in date_cols.items():
    df = tables[name]          # the same object as the variable of that name, so the change is visible everywhere
    for c in cols:
        before = df[c].isna().sum()
        df[c] = pd.to_datetime(df[c], errors="coerce")
        rows.append({"table": name, "column": c, "missing before": before,
                     "new missing after conversion": df[c].isna().sum() - before,
                     "earliest": df[c].min(), "latest": df[c].max()})
pd.DataFrame(rows)

player_injuries      rows:   143,195   columns: 7
player_performances  rows: 1,878,719   columns: 20
players              rows:    50,149   columns: 26
appearances          rows: 1,894,350   columns: 13
games                rows:    88,958   columns: 23
transfers            rows:   175,165   columns: 10


,table,column,missing before,new missing after conversion,earliest,latest
0,player_injuries,from_date,22,0,1973-09-02,2025-12-22
1,player_injuries,end_date,1523,0,1973-09-06,2026-09-01
2,appearances,date,0,0,2012-07-03,2026-06-28
3,games,date,0,0,2006-06-09,2026-07-06
4,transfers,transfer_date,0,0,1993-07-01,2030-06-30
5,players,date_of_birth,49,0,1968-07-31,2011-02-23


### 0.3 Rules and expected numbers

The rules are the ones fixed in `01_data_audit.ipynb`. They are written once, here, as constants. `EXPECTED` holds the numbers of injuries and players that the audit found after each step of the funnel; the notebook stops when a step gives something else.

In [2]:
# 0.3 Rules of the cohort (fixed in 01_data_audit.ipynb)
TOP5 = ["GB1", "ES1", "IT1", "L1", "FR1"]            # Premier League, La Liga, Serie A, Bundesliga, Ligue 1
MENISCUS_LABELS = ["Meniscus damage", "Meniscus tear", "Meniscus injury",
                   "Tear of the lateral meniscus", "Meniscus irritation"]

PRE_MONTHS = 12            # window before from_date (appearances)
POST_MONTHS = 24           # window after end_date (appearances)
POST_SHORT_MONTHS = 12    # shorter window after end_date: B and change of club (appearances)
ACL_DAYS = 30              # no cruciate ligament injury within this many days of the meniscal injury
MIN_TOP5_MINUTES = 1       # level criterion: at least this many minutes in a top-5 league in the 12 months before
MIN_DAYS_MISSED = 29       # serious injury: absence of 29 days or more (Fuller et al. 2006)
OTHER_AFTER_DAYS = 90      # flag: another injury within this many days after end_date
# (injuries, players) after each step of the funnel, as found in the audit
EXPECTED = {
    "0 all meniscal injuries":                      (1863, 1657),
    "1 start and end date present":                 (1838, 1632),
    "2 both windows inside appearances":            (1277, 1164),
    "3 no cruciate ligament injury within 30 days": (1248, 1138),
    "4 level criterion: minutes in the top 5 before": (195, 181),
    "5 absence of 29 days or more":                 (169, 158),
    "6 one index injury per player":                (158, 158),
}

### 0.4 The funnel table

After each step we record the number of injuries and the number of players, and what was lost against the previous step. A player is lost only when all of his injuries are gone, so the lost players do not add up to the lost injuries. Step 0 takes every injury with one of the five meniscal labels.

In [3]:
# 0.4 The funnel table: injuries and players after each step, and what was lost against the previous step
funnel = []

def log_step(name, rule, df):
    k = int(name.split()[0])                                              # the number of the step
    funnel[:] = [r for r in funnel if int(r["step"].split()[0]) < k]      # running a cell again replaces its row
    n_inj, n_pl = len(df), df["player_id"].nunique()
    assert (n_inj, n_pl) == EXPECTED[name], f"{name}: {(n_inj, n_pl)} (injuries, players) instead of {EXPECTED[name]} in the audit"
    prev = funnel[-1] if funnel else {"injuries": n_inj, "players": n_pl}
    funnel.append({"step": name, "rule": rule, "injuries": n_inj, "players": n_pl,
                   "lost injuries": prev["injuries"] - n_inj, "lost players": prev["players"] - n_pl})

# step 0: all meniscal injuries (the five labels of the audit)
meniscus = player_injuries[player_injuries["injury_reason"].isin(MENISCUS_LABELS)]
log_step("0 all meniscal injuries", "injury_reason is one of the five meniscal labels", meniscus)
pd.DataFrame(funnel)

,step,rule,injuries,players,lost injuries,lost players
0,0 all meniscal injuries,injury_reason is one of the five meniscal labels,1863,1657,0,0


## Step 1: Cleaning

### 1.1 Injuries without dates, and duplicates

An injury needs a start and an end date to have windows. We drop the injuries without one of them and any exact duplicate (same player, dates and label); the audit found 25 injuries without a date and no duplicates. Each remaining injury gets `inj_id`, the number of its row in the cleaned table.

In [4]:
# 1.1 Cleaning: injuries without a start or end date, and duplicates
KEY = ["player_id", "from_date", "end_date", "injury_reason"]

no_dates = meniscus[["from_date", "end_date"]].isna().any(axis=1).sum()
with_dates = meniscus.dropna(subset=["from_date", "end_date"])
duplicates = with_dates.duplicated(KEY).sum()
print("injuries without a start or end date:", no_dates, "| full duplicates:", duplicates)

m = with_dates.drop_duplicates(KEY).reset_index(drop=True)
m["inj_id"] = m.index                      # the number of the row in the cleaned table
log_step("1 start and end date present", "from_date and end_date both present, duplicates removed", m)
pd.DataFrame(funnel)

injuries without a start or end date: 25 | full duplicates: 0


,step,rule,injuries,players,lost injuries,lost players
0,0 all meniscal injuries,injury_reason is one of the five meniscal labels,1863,1657,0,0
1,1 start and end date present,"from_date and end_date both present, duplicates removed",1838,1632,25,25


### 1.2 Seasons and the summer flag

A season runs from July to June and is named by the year it starts. `inj_season` is the season of `from_date`, `ret_season` the season of `end_date`. `summer_start` marks injuries that start in May-July, between seasons: `days_missed` also counts the break, so these injuries can be long in days and short in matches.

In [5]:
# 1.2 Seasons of the injury and of the return, and the summer flag
def european_season(dates):
    return np.where(dates.dt.month >= 7, dates.dt.year, dates.dt.year - 1)      # a season runs from July to June

m["inj_season"] = european_season(m["from_date"])            # season in which the injury happened
m["ret_season"] = european_season(m["end_date"])             # season in which the player returned
m["summer_start"] = m["from_date"].dt.month.isin([5, 6, 7])  # the injury starts between seasons

m[["player_id", "from_date", "end_date", "inj_season", "ret_season", "summer_start"]].head()

,player_id,from_date,end_date,inj_season,ret_season,summer_start
0,100127,2012-01-10,2012-03-10,2011,2011,False
1,10076,2014-10-17,2015-02-25,2014,2014,False
2,100988,2022-10-05,2023-08-06,2022,2023,False
3,101253,2018-03-18,2018-09-18,2017,2018,False
4,10197,1998-11-25,1999-01-03,1998,1998,False


## Step 2: Windows inside `appearances`

### 2.1 Both windows inside the period covered by `appearances`

Every player needs data before and after the injury, so both windows must lie inside the period covered by `appearances`: 12 months before `from_date` and 24 months after `end_date`. We also store the end of the shorter window, 12 months after `end_date`, used for B and for the change of club.

In [6]:
# 2.1 Both windows inside the period covered by appearances
m["pre_start"]  = m["from_date"] - pd.DateOffset(months=PRE_MONTHS)
m["post12_end"] = m["end_date"] + pd.DateOffset(months=POST_SHORT_MONTHS)
m["post_end"]   = m["end_date"] + pd.DateOffset(months=POST_MONTHS)

app_min, app_max = appearances["date"].min(), appearances["date"].max()
print("appearances covers", app_min.date(), "to", app_max.date())
m["in_range"] = (m["pre_start"] >= app_min) & (m["post_end"] <= app_max)

c2 = m[m["in_range"]]
log_step("2 both windows inside appearances", "12 months before from_date and 24 months after end_date inside appearances", c2)
pd.DataFrame(funnel)

appearances covers 2012-07-03 to 2026-06-28


,step,rule,injuries,players,lost injuries,lost players
0,0 all meniscal injuries,injury_reason is one of the five meniscal labels,1863,1657,0,0
1,1 start and end date present,"from_date and end_date both present, duplicates removed",1838,1632,25,25
2,2 both windows inside appearances,12 months before from_date and 24 months after end_date inside appearances,1277,1164,561,468


## Step 3: Other injuries near the meniscal injury

### 3.1 Cruciate ligament injury within 30 days, and two flags

Other injuries can distort what we see in the windows. A cruciate ligament injury that starts from 30 days before `from_date` to 30 days after `end_date` (`acl_30d`) removes the injury from the cohort: we could not tell the effect of the meniscus from the effect of the ligament. Two more flags remove nobody and are kept for the sensitivity analyses: `other_during` (another non-meniscal injury that starts during the absence) and `other_90d_after` (another injury that starts within 90 days after `end_date`). `other_injury` is true when either of them is true.

In [7]:
# 3.1 Cruciate ligament injury within 30 days (leaves the cohort) and two flags for other injuries (nobody leaves)
acl_labels = player_injuries.loc[player_injuries["injury_reason"].str.contains("cruciate", case=False, na=False), "injury_reason"].unique()
other = player_injuries[~player_injuries["injury_reason"].isin(MENISCUS_LABELS)].dropna(subset=["from_date"])

x = m[["inj_id", "player_id", "from_date", "end_date"]].merge(
    other[["player_id", "injury_reason", "from_date"]].rename(columns={"injury_reason": "o_reason", "from_date": "o_from"}),
    on="player_id")

acl_near = (x["o_reason"].isin(acl_labels) & (x["o_from"] >= x["from_date"] - pd.Timedelta(days=ACL_DAYS))
            & (x["o_from"] <= x["end_date"] + pd.Timedelta(days=ACL_DAYS)))
during  = (x["o_from"] >= x["from_date"]) & (x["o_from"] <= x["end_date"])
after90 = (x["o_from"] > x["end_date"]) & (x["o_from"] <= x["end_date"] + pd.Timedelta(days=OTHER_AFTER_DAYS))

m["acl_30d"]         = m["inj_id"].isin(x.loc[acl_near, "inj_id"])
m["other_during"]    = m["inj_id"].isin(x.loc[during, "inj_id"])
m["other_90d_after"] = m["inj_id"].isin(x.loc[after90, "inj_id"])
m["other_injury"]    = m["other_during"] | m["other_90d_after"]

print("ACL labels in the data:", list(acl_labels))
display(pd.DataFrame({"all meniscal injuries (m)": m[["acl_30d", "other_during", "other_90d_after"]].sum(),
                      "windows inside appearances": m.loc[m["in_range"], ["acl_30d", "other_during", "other_90d_after"]].sum()}))

c3 = m[m["in_range"] & ~m["acl_30d"]]
log_step("3 no cruciate ligament injury within 30 days", "no cruciate ligament injury from 30 days before from_date to 30 days after end_date", c3)
pd.DataFrame(funnel)

ACL labels in the data: ['Cruciate ligament tear', 'Cruciate ligament injury', 'Cruciate ligament surgery', 'Cruciate ligament strain', 'Partial damage to the cruciate ligament']


,all meniscal injuries (m),windows inside appearances
acl_30d,42,29
other_during,116,77
other_90d_after,301,207


,step,rule,injuries,players,lost injuries,lost players
0,0 all meniscal injuries,injury_reason is one of the five meniscal labels,1863,1657,0,0
1,1 start and end date present,"from_date and end_date both present, duplicates removed",1838,1632,25,25
2,2 both windows inside appearances,12 months before from_date and 24 months after end_date inside appearances,1277,1164,561,468
3,3 no cruciate ligament injury within 30 days,no cruciate ligament injury from 30 days before from_date to 30 days after end_date,1248,1138,29,26


## Step 4: The level criterion

### 4.1 Minutes in a top-5 league in the 12 months before the injury

The project asks about players of Europe's top five leagues, so the player must have played there before the injury: at least 1 minute (`MIN_TOP5_MINUTES`) in a top-5 league match in the 12 months before `from_date` (`appearances`, matches from `pre_start` up to the day before `from_date`). This is the *level criterion*; the size of the baseline is a separate matter and becomes the column `baseline_group` later. `min_top5_pre` keeps the minutes for every injury.

In [8]:
# 4.1 Level criterion: minutes in a top-5 league in the 12 months before from_date (appearances)
a5 = appearances.loc[appearances["player_id"].isin(m["player_id"]) & appearances["competition_id"].isin(TOP5),
                     ["player_id", "date", "minutes_played"]]
x5 = m[["inj_id", "player_id", "pre_start", "from_date"]].merge(a5, on="player_id")
x5 = x5[(x5["date"] >= x5["pre_start"]) & (x5["date"] < x5["from_date"])]
m["min_top5_pre"] = m["inj_id"].map(x5.groupby("inj_id")["minutes_played"].sum()).fillna(0)
m["top5_pre"] = m["min_top5_pre"] >= MIN_TOP5_MINUTES

c4 = m.loc[c3.index]                      # c3 was cut before top5_pre existed, so take its rows from m again
c4 = c4[c4["top5_pre"]]
log_step("4 level criterion: minutes in the top 5 before", f"at least {MIN_TOP5_MINUTES} minute in a top-5 league in the 12 months before from_date", c4)
pd.DataFrame(funnel)

,step,rule,injuries,players,lost injuries,lost players
0,0 all meniscal injuries,injury_reason is one of the five meniscal labels,1863,1657,0,0
1,1 start and end date present,"from_date and end_date both present, duplicates removed",1838,1632,25,25
2,2 both windows inside appearances,12 months before from_date and 24 months after end_date inside appearances,1277,1164,561,468
3,3 no cruciate ligament injury within 30 days,no cruciate ligament injury from 30 days before from_date to 30 days after end_date,1248,1138,29,26
4,4 level criterion: minutes in the top 5 before,at least 1 minute in a top-5 league in the 12 months before from_date,195,181,1053,957


## Step 5: Serious injury

### 5.1 Absence of 29 days or more

A serious injury is one with an absence of more than 28 days (Fuller et al. 2006), so `days_missed` of 29 or more; `days_missed` counts days inclusively. A stricter threshold (60 days) is a sensitivity analysis done later by filtering the cohort, not by building it again.

In [9]:
# 5.1 Serious injury: absence of 29 days or more
c5 = c4[c4["days_missed"] >= MIN_DAYS_MISSED]
log_step("5 absence of 29 days or more", f"days_missed is {MIN_DAYS_MISSED} or more (more than 28 days, Fuller et al. 2006)", c5)
pd.DataFrame(funnel)

,step,rule,injuries,players,lost injuries,lost players
0,0 all meniscal injuries,injury_reason is one of the five meniscal labels,1863,1657,0,0
1,1 start and end date present,"from_date and end_date both present, duplicates removed",1838,1632,25,25
2,2 both windows inside appearances,12 months before from_date and 24 months after end_date inside appearances,1277,1164,561,468
3,3 no cruciate ligament injury within 30 days,no cruciate ligament injury from 30 days before from_date to 30 days after end_date,1248,1138,29,26
4,4 level criterion: minutes in the top 5 before,at least 1 minute in a top-5 league in the 12 months before from_date,195,181,1053,957
5,5 absence of 29 days or more,"days_missed is 29 or more (more than 28 days, Fuller et al. 2006)",169,158,26,23


## Step 6: The cohort W

### 6.1 One index injury per player

A player can have several injuries that pass all the steps. To have one row per player we keep the earliest one, the index injury; when two injuries of a player start on the same day, the one with the lower `inj_id`. The other injuries of that player are not used. The result is the cohort W.

In [10]:
# 6.1 One index injury per player: the earliest
several = int((c5.groupby("player_id").size() > 1).sum())
print("players with more than one injury at this point:", several)

W = c5.sort_values(["from_date", "inj_id"]).drop_duplicates("player_id").copy()      # ties on from_date: the lower inj_id
assert W["player_id"].is_unique
log_step("6 one index injury per player", "the earliest injury of each player (the index injury)", W)
pd.DataFrame(funnel)

players with more than one injury at this point: 11


,step,rule,injuries,players,lost injuries,lost players
0,0 all meniscal injuries,injury_reason is one of the five meniscal labels,1863,1657,0,0
1,1 start and end date present,"from_date and end_date both present, duplicates removed",1838,1632,25,25
2,2 both windows inside appearances,12 months before from_date and 24 months after end_date inside appearances,1277,1164,561,468
3,3 no cruciate ligament injury within 30 days,no cruciate ligament injury from 30 days before from_date to 30 days after end_date,1248,1138,29,26
4,4 level criterion: minutes in the top 5 before,at least 1 minute in a top-5 league in the 12 months before from_date,195,181,1053,957
5,5 absence of 29 days or more,"days_missed is 29 or more (more than 28 days, Fuller et al. 2006)",169,158,26,23
6,6 one index injury per player,the earliest injury of each player (the index injury),158,158,11,0


In [11]:
W.head()

,player_id,season_name,injury_reason,from_date,end_date,days_missed,games_missed,inj_id,inj_season,ret_season,summer_start,pre_start,post12_end,post_end,in_range,acl_30d,other_during,other_90d_after,other_injury,min_top5_pre,top5_pre
827,34593,13/14,Tear of the lateral meniscus,2013-08-12,2014-02-13,186.0,38,827,2013,2013,False,2012-08-12,2015-02-13,2016-02-13,True,False,False,False,False,2189.0,True
397,20003,13/14,Meniscus injury,2013-09-09,2013-10-15,37.0,7,397,2013,2013,False,2012-09-09,2014-10-15,2015-10-15,True,False,False,True,True,386.0,True
1655,77834,13/14,Meniscus injury,2013-10-15,2013-12-01,48.0,11,1655,2013,2013,False,2012-10-15,2014-12-01,2015-12-01,True,False,False,True,True,2644.0,True
455,21829,13/14,Meniscus damage,2013-11-20,2014-04-13,145.0,28,455,2013,2013,False,2012-11-20,2015-04-13,2016-04-13,True,False,False,True,True,360.0,True
1105,45403,13/14,Meniscus damage,2013-11-22,2014-02-06,77.0,23,1105,2013,2013,False,2012-11-22,2015-02-06,2016-02-06,True,False,False,False,False,2966.0,True


## Step 7: Columns of the cohort W

The funnel is finished; from here we add the columns that the analysis needs, group by group.

### 7.1 Baseline before the injury

The baseline of a player is what he played in the 12 months before `from_date` (`appearances`). `min_top5_pre` (top-5 leagues) is the level criterion. `min_pre` counts the minutes in all 14 domestic leagues that `appearances` covers, the same leagues as the outcome B. `baseline_group` says how reliable the baseline is: 1-269 minutes (fewer than three full matches), 270-899 and 900+ (about ten full matches or more); it is built from the top-5 minutes.

In [12]:
# 7.1 Baseline before the injury: minutes in the top 5 and in the 14 domestic leagues, baseline group
comp_type = games.drop_duplicates("competition_id").set_index("competition_id")["competition_type"]
L14 = {c for c in set(appearances["competition_id"]) if comp_type.get(c) == "domestic_league"}     # the leagues that appearances covers
assert len(L14) == 14 and set(TOP5) <= L14

a14 = appearances.loc[appearances["player_id"].isin(W["player_id"]) & appearances["competition_id"].isin(L14),
                      ["player_id", "date", "minutes_played"]]
x14 = W[["inj_id", "player_id", "pre_start", "from_date"]].merge(a14, on="player_id")
x14 = x14[(x14["date"] >= x14["pre_start"]) & (x14["date"] < x14["from_date"])]
W["min_pre"] = W["inj_id"].map(x14.groupby("inj_id")["minutes_played"].sum()).fillna(0)

W["baseline_group"] = pd.cut(W["min_top5_pre"], [0, 269, 899, 10**9], labels=["1-269", "270-899", "900+"])    # by minutes in the top 5

display(W["baseline_group"].value_counts().sort_index().rename("players").to_frame())
print("minutes in the 14 leagues are never below the minutes in the top 5:", bool((W["min_pre"] >= W["min_top5_pre"]).all()),
      "| players with more minutes in the 14 leagues:", int((W["min_pre"] > W["min_top5_pre"]).sum()))

,players
baseline_group,
1-269,30
270-899,43
900+,85


minutes in the 14 leagues are never below the minutes in the top 5: True | players with more minutes in the 14 leagues: 11


### 7.2 Age and position

Age at `from_date` and the position come from the table `players` (one row per player, so the mapping cannot add rows).

In [13]:
# 7.2 Age at the injury and position (table players)
pl = players.set_index("player_id")
W["age_at_injury"] = (W["from_date"] - W["player_id"].map(pl["date_of_birth"])).dt.days / 365.25
W["position"] = W["player_id"].map(pl["position"])

print("players with age and position:", int((W["age_at_injury"].notna() & W["position"].notna()).sum()), "of", len(W))
print("age at injury, percentiles 10/25/50/75/90:", W["age_at_injury"].quantile([.1, .25, .5, .75, .9]).round(1).tolist())
print("positions:", W["position"].value_counts().to_dict())

players with age and position: 158 of 158
age at injury, percentiles 10/25/50/75/90: [20.8, 22.5, 25.4, 28.9, 32.1]
positions: {'Defender': 56, 'Midfield': 44, 'Attack': 41, 'Goalkeeper': 17}


### 7.3 Minutes after the return

The outcome B compares the minutes after the return with the minutes before, in the same 14 domestic leagues, no cups (`appearances`). `min_post_12` and `min_post_24` are the minutes in the 12 and 24 months after `end_date`. `l14_post12` and `l14_post24` say whether the player has at least one such league match in that window; a player without one has no minutes to measure.

In [14]:
# 7.3 Minutes in the 14 domestic leagues after the return: 12 and 24 months from end_date (appearances)
a14w = appearances.loc[appearances["player_id"].isin(W["player_id"]) & appearances["competition_id"].isin(L14),
                       ["player_id", "date", "minutes_played"]]
xp = W[["inj_id", "player_id", "end_date", "post12_end", "post_end"]].merge(a14w, on="player_id")
xp = xp[xp["date"] > xp["end_date"]]
p12m, p24m = xp[xp["date"] <= xp["post12_end"]], xp[xp["date"] <= xp["post_end"]]

W["min_post_12"] = W["inj_id"].map(p12m.groupby("inj_id")["minutes_played"].sum()).fillna(0)
W["min_post_24"] = W["inj_id"].map(p24m.groupby("inj_id")["minutes_played"].sum()).fillna(0)
W["l14_post12"] = W["inj_id"].isin(p12m["inj_id"])          # at least one league match (14 leagues) in the 12 months after
W["l14_post24"] = W["inj_id"].isin(p24m["inj_id"])

t = W.groupby("baseline_group", observed=True).agg(players=("inj_id", "size"), league_match_12m=("l14_post12", "sum"), league_match_24m=("l14_post24", "sum"))
t.loc["all"] = t.sum()
display(t)
print("a league match but 0 minutes in the 12 months after:", int((W["l14_post12"] & (W["min_post_12"] == 0)).sum()))

,players,league_match_12m,league_match_24m
baseline_group,,,
1-269,30,16,17
270-899,43,32,32
900+,85,77,79
all,158,125,128


a league match but 0 minutes in the 12 months after: 0


### 7.4 First match after the return

`first_match` is the date of the first match in any competition (league or cup) after `end_date`, `days_to_first` the number of days to it (`appearances`, 24 months from `end_date`). `has12` and `has24` say whether the player has a match in the 12 and in the 24 months. A player without a match has an empty `first_match` and `days_to_first`: it is a separate category, not a zero.

In [15]:
# 7.4 First match after the return, in any competition (appearances)
aw = appearances[appearances["player_id"].isin(W["player_id"])][["player_id", "date"]]
xa = W[["inj_id", "player_id", "end_date", "post12_end", "post_end"]].merge(aw, on="player_id")
xa = xa[(xa["date"] > xa["end_date"]) & (xa["date"] <= xa["post_end"])]

W["first_match"] = W["inj_id"].map(xa.groupby("inj_id")["date"].min())
W["days_to_first"] = (W["first_match"] - W["end_date"]).dt.days             # empty when there is no match in the 24 months
W["has12"] = W["inj_id"].isin(xa.loc[xa["date"] <= xa["post12_end"], "inj_id"])
W["has24"] = W["first_match"].notna()

t = W.groupby("baseline_group", observed=True).agg(players=("inj_id", "size"), no_match_12m=("has12", lambda s: (~s).sum()), no_match_24m=("has24", lambda s: (~s).sum()))
t.loc["all"] = t.sum()
display(t)
print("days from end_date to the first match, percentiles 10/25/50/75/90 (players with a match):", W["days_to_first"].quantile([.1, .25, .5, .75, .9]).round(0).astype(int).tolist())

,players,no_match_12m,no_match_24m
baseline_group,,,
1-269,30,14,13
270-899,43,9,9
900+,85,8,6
all,158,31,28


days from end_date to the first match, percentiles 10/25/50/75/90 (players with a match): [2, 5, 15, 56, 118]


### 7.5 Change of club

`club_pre` is the club of the last match in the 12 months before `from_date` (`appearances`, `player_club_id`). `club_changed` is true when the player played at least one match (any competition) for a different club in the 12 months after `end_date`. It is empty for players without a match in that window. `transfer_in_window` marks a transfer in `transfers` between `from_date` and 12 months after `end_date`; it is only a check, because `transfers` is incomplete.

In [16]:
# 7.5 Change of club (appearances; transfers only as a check)
xc = W[["inj_id", "player_id", "pre_start", "from_date", "end_date", "post12_end"]].merge(
    appearances[["player_id", "date", "player_club_id"]], on="player_id")
last_pre = (xc[(xc["date"] >= xc["pre_start"]) & (xc["date"] < xc["from_date"])]
            .sort_values("date").groupby("inj_id")["player_club_id"].last())
W["club_pre"] = W["inj_id"].map(last_pre)                  # the club of the last match in the 12 months before from_date
assert W["club_pre"].notna().all()                         # every player of W has a match before (level criterion)

p12 = xc[(xc["date"] > xc["end_date"]) & (xc["date"] <= xc["post12_end"])].merge(W[["inj_id", "club_pre"]], on="inj_id")
changed = W["inj_id"].isin(p12.loc[p12["player_club_id"] != p12["club_pre"], "inj_id"])
W["club_changed"] = changed.astype("boolean").where(W["has12"])      # empty when there is no match in the 12 months after

tw = W[["inj_id", "player_id", "from_date", "post12_end"]].merge(transfers[["player_id", "transfer_date"]], on="player_id")
tw = tw[(tw["transfer_date"] > tw["from_date"]) & (tw["transfer_date"] <= tw["post12_end"])]
W["transfer_in_window"] = W["inj_id"].isin(tw["inj_id"])             # a check only

print("players with a match in the 12 months after:", int(W["has12"].sum()), "| club changed:", int(W["club_changed"].sum()),
      "| players of W present in transfers:", int(W["player_id"].isin(transfers["player_id"]).sum()), "of", len(W))
print(f"share of club changes: {W['club_changed'].mean():.1%} (it would be {W['club_changed'].fillna(False).astype(bool).mean():.1%} if the empty values were False)")
display(W.groupby("baseline_group", observed=True)["club_changed"].agg(["sum", "count"]).rename(columns={"sum": "club changed", "count": "with a match in 12 months"}))
w12 = W[W["has12"]]
pd.crosstab(w12["club_changed"].astype(bool).rename("club changed (appearances)"), w12["transfer_in_window"].rename("transfer in the window (transfers)"))

players with a match in the 12 months after: 127 | club changed: 45 | players of W present in transfers: 91 of 158
share of club changes: 35.4% (it would be 28.5% if the empty values were False)


,club changed,with a match in 12 months
baseline_group,,
1-269,6,16
270-899,15,34
900+,24,77


transfer in the window (transfers),False,True
club changed (appearances),,
False,74,8
True,15,30


### 7.6 Level after the return: the dictionary of leagues

The level after the return is a label for the competitions where the player played in the 24 months after the return (rules fixed in `01_data_audit.ipynb`, step 6). `appearances` sees the top 5 and nine first divisions; the second and third divisions of the top-5 countries and fourteen other first divisions are visible only in `player_performances`. The dictionary below lists them by competition id. `season_start` is the year in which a season of `player_performances` starts.

In [17]:
# 7.6 Level after the return, part 1: the dictionary of leagues and the season numbers of player_performances
SECOND_THIRD = {"ES2", "IT2", "GB2", "L2", "FR2", "L3", "FR3", "GB3", "IT3A", "IT3B", "IT3C", "ES3A"}   # 2nd and 3rd divisions of the five countries
FIRST_OTHER = {"TR1", "NL1", "BE1", "PO1", "A1", "RU1", "SC1", "C1", "UKR1", "DK1", "SE1", "PL1", "GR1", "TS1", "BU1",
               "MLS1", "MEX1", "MEXA", "SA1", "AUS1", "BRA1", "CSL", "QSL"}                                   # first divisions elsewhere

VISIBLE = set(appearances["competition_id"])
FIRST_VIS = FIRST_OTHER & VISIBLE             # first divisions outside the top 5 that appearances contains
FIRST_NOTVIS = FIRST_OTHER - VISIBLE          # first divisions visible only in player_performances
assert L14 == set(TOP5) | FIRST_VIS           # the same 14 leagues as in 7.1

def season_start(name):
    name = str(name)
    if len(name) == 4:                        # written as "2019"
        return int(name)
    yy = int(name[:2])                        # written as "19/20"
    return 1900 + yy if yy >= 50 else 2000 + yy

player_performances["season_start"] = player_performances["season_name"].map({s: season_start(s) for s in player_performances["season_name"].unique()})
print("first divisions outside the top 5: visible in appearances", len(FIRST_VIS), "| only in player_performances", len(FIRST_NOTVIS))
print("seasons in player_performances:", int(player_performances["season_start"].min()), "to", int(player_performances["season_start"].max()))

first divisions outside the top 5: visible in appearances 9 | only in player_performances 14
seasons in player_performances: 1960 to 2026


### 7.7 League matches per level, and the flag `played_top5`

Two sources, two windows. `appearances`: the 24 months from `end_date`, exact dates; it gives the matches in a top-5 league (level 1) and in the other first divisions it contains (part of level 3). `player_performances`: the two seasons after the season of the return (the season of the return itself is not counted), rows with at least one appearance; it gives the second and third divisions (level 2) and the first divisions that `appearances` does not contain (the rest of level 3). `top5_matches` and the flag `played_top5` come from `appearances` only, so they do not depend on the seasonal window.

In [18]:
# 7.7 Level after the return, part 2: league matches per level in the two sources, and the flag played_top5
inj = list(W["inj_id"])

# appearances: matches in the 24 months from end_date
xw = W[["inj_id", "player_id", "end_date", "post_end"]].merge(appearances[["player_id", "date", "competition_id"]], on="player_id")
aw = xw[(xw["date"] > xw["end_date"]) & (xw["date"] <= xw["post_end"])]
n_top5  = aw[aw["competition_id"].isin(TOP5)].groupby("inj_id").size().reindex(inj).fillna(0)        # matches in a top-5 league
n_first = aw[aw["competition_id"].isin(FIRST_VIS)].groupby("inj_id").size().reindex(inj).fillna(0)   # matches in another visible league
n_any   = aw.groupby("inj_id").size().reindex(inj).fillna(0)                                         # matches in any competition

# player_performances: rows with at least one appearance in the two seasons after the season of the return
yw = W[["inj_id", "player_id", "ret_season"]].merge(
    player_performances[["player_id", "season_start", "competition_id", "nb_on_pitch"]], on="player_id")
yw_all = yw[(yw["season_start"] > yw["ret_season"]) & (yw["season_start"] <= yw["ret_season"] + 2)]     # includes rows with 0 appearances
yw = yw_all[yw_all["nb_on_pitch"] > 0]

W["top5_matches"] = W["inj_id"].map(n_top5).astype(int)      # appearances, 24 months from end_date
W["played_top5"] = W["top5_matches"] > 0

view = np.select([n_top5 > 0, n_first > 0, n_any > 0],
                 ["a match in a top-5 league", "no top 5, a match in another visible league", "only cups or other competitions"],
                 default="no match in appearances")
display(pd.Series(view).value_counts().rename("players").to_frame())
print("played_top5 is True for", int(W["played_top5"].sum()), "players; median number of top-5 matches among them:", W.loc[W["played_top5"], "top5_matches"].median())

,players
a match in a top-5 league,116
no match in appearances,28
"no top 5, a match in another visible league",12
only cups or other competitions,2


played_top5 is True for 116 players; median number of top-5 matches among them: 34.0


### 7.8 Level after the return: two rules

*H*: the first level in the order 1, 2, 3 with at least one league match. *M*: the level with the most league matches (tie: the lower number). Levels are labels, not a quality scale: 1 top 5; 2 second or third division of a top-5 country; 3 first division elsewhere; 4 other competitions only; 5 no league match, split into 5a (in a squad, no appearances: a row exists in `player_performances`) and 5b (no data). `level_after` follows rule M; `level_H` and `level_M` are stored for the comparison. Cups and qualifiers never set the level.

In [19]:
# 7.8 Level after the return, part 3: rule H, rule M and the level used
p_second  = yw[yw["competition_id"].isin(SECOND_THIRD)].groupby("inj_id")["nb_on_pitch"].sum().reindex(inj).fillna(0)
p_firstnv = yw[yw["competition_id"].isin(FIRST_NOTVIS)].groupby("inj_id")["nb_on_pitch"].sum().reindex(inj).fillna(0)
p_any     = yw.groupby("inj_id")["nb_on_pitch"].sum().reindex(inj).fillna(0)

LVL = ["1 top 5", "2 second/third division, top-5 country", "3 first division elsewhere"]
C = pd.DataFrame({LVL[0]: n_top5, LVL[1]: p_second, LVL[2]: n_first + p_firstnv})      # league matches per level, both sources
OTHER, NONE = "4 other competitions only", "5 no league match"
other_any = (n_any + p_any) > 0

def rule_H(r, i):                        # highest level with at least one match
    for col in LVL:
        if r[col] > 0: return col
    return OTHER if other_any[i] else NONE

def rule_M(r, i):                        # level with the most league matches (tie: the lower number)
    if r.sum() > 0:
        best = r.max()
        return next(col for col in LVL if r[col] == best)
    return OTHER if other_any[i] else NONE

H = pd.Series({i: rule_H(C.loc[i], i) for i in inj})
M = pd.Series({i: rule_M(C.loc[i], i) for i in inj})

# level 5 splits into "in a squad, no appearances" (a row exists) and "no data" (no match, no row)
has_row = set(yw_all["inj_id"])
def final_label(i):
    return M[i] if M[i] != NONE else ("5a in squad, no appearances" if i in has_row else "5b no data")

W["level_H"] = W["inj_id"].map(H)
W["level_M"] = W["inj_id"].map(M)
W["level_after"] = W["inj_id"].map(final_label)

display(pd.DataFrame({"H (highest level)": H.value_counts(), "M (most matches)": M.value_counts()}).fillna(0).astype(int).sort_index())
display(W["level_after"].value_counts().sort_index().rename("players (rule M, level 5 split)").to_frame())
print("H and M give the same level for", int((H == M).sum()), "of", len(W), "players and differ for", int((H != M).sum()))
print("played_top5 is True for", int(W["played_top5"].sum()), "players; for", int((W["played_top5"] & (W["level_M"] != LVL[0])).sum()), "of them rule M gives a level below the top 5")

,H (highest level),M (most matches)
1 top 5,116,86
"2 second/third division, top-5 country",23,42
3 first division elsewhere,14,25
5 no league match,5,5


,"players (rule M, level 5 split)"
level_after,
1 top 5,86
"2 second/third division, top-5 country",42
3 first division elsewhere,25
"5a in squad, no appearances",1
5b no data,4


H and M give the same level for 127 of 158 players and differ for 31
played_top5 is True for 116 players; for 30 of them rule M gives a level below the top 5


### 7.9 How stable is the level?

Rule M adds up counts from two windows. To see how much this matters we repeat rule M with every level counted from `player_performances` only (one source, one window). `level_M_one_source` holds that level, and `level_unstable` is true when it differs from `level_M`. An unstable player whose one-source level is 4 or 5 is an artefact of the seasonal window (his matches lie in the season of the return, which the window skips); the others changed level for real.

In [20]:
# 7.9 Level after the return, part 4: the same rule M with every level counted from player_performances only
q = lambda ids: yw[yw["competition_id"].isin(ids)].groupby("inj_id")["nb_on_pitch"].sum().reindex(inj).fillna(0)
Cp = pd.DataFrame({LVL[0]: q(TOP5), LVL[1]: q(SECOND_THIRD), LVL[2]: q(FIRST_VIS | FIRST_NOTVIS)})
Mp = Cp.apply(lambda r: next(col for col in LVL if r[col] == r.max()) if r.sum() > 0 else (OTHER if p_any[r.name] > 0 else NONE), axis=1)

W["level_M_one_source"] = W["inj_id"].map(Mp)
W["level_unstable"] = W["level_M_one_source"] != W["level_M"]

diff = Mp != M
print("level differs between the two ways for", int(W["level_unstable"].sum()), "of", len(W), "players")
print("of them with no league match in the seasonal window (all matches in the season of the return):", int((diff & (Cp.sum(axis=1) == 0)).sum()))
print("of them a real change of level:", int((diff & (Cp.sum(axis=1) > 0)).sum()))
print("of the unstable players played_top5 is True for", int(W.loc[W["level_unstable"], "played_top5"].sum()))

level differs between the two ways for 11 of 158 players
of them with no league match in the seasonal window (all matches in the season of the return): 5
of them a real change of level: 6
of the unstable players played_top5 is True for 11


### 7.10 Flags for the sensitivity analyses, and the absence band

`not_first_meniscus` marks an index injury that is not the first meniscal injury of the player in the data (`n_men_before` counts the earlier ones in the cleaned table of all 1,838 injuries). `few_games_missed` marks an absence with 2 or fewer games missed: a long absence in days that cost almost no matches. Together with `summer_start`, `other_during`, `other_90d_after` and `other_injury` (made earlier) these flags let each sensitivity analysis be a filter. `absence_band` only describes the absence (29-42, 43-60, 61-90, 91+ days); the analysis with a stricter threshold filters on `days_missed`.

In [21]:
# 7.10 Flags for the sensitivity analyses, and the absence band
order = m.sort_values(["player_id", "from_date", "inj_id"])
earlier = order.groupby("player_id").cumcount().reindex(m.index)         # earlier meniscal injuries of the same player in the cleaned data
W["n_men_before"] = W["inj_id"].map(earlier)
W["not_first_meniscus"] = W["n_men_before"] > 0
W["few_games_missed"] = W["games_missed"] <= 2                           # a long absence with almost no matches missed
W["absence_band"] = pd.cut(W["days_missed"], [28, 42, 60, 90, 10**5], labels=["29-42", "43-60", "61-90", "91+"])      # description only

print("flags in W:", {f: int(W[f].sum()) for f in ["summer_start", "other_during", "other_90d_after", "other_injury", "not_first_meniscus", "few_games_missed"]})
display(W["absence_band"].value_counts().sort_index().rename("players").to_frame())

flags in W: {'summer_start': 20, 'other_during': 11, 'other_90d_after': 42, 'other_injury': 51, 'not_first_meniscus': 12, 'few_games_missed': 5}


,players
absence_band,
29-42,39
43-60,46
61-90,28
91+,45


### 7.11 Which measures are possible for each player

For each player we record whether the outcome can be measured, without looking at any value. `B_minutes_possible`: the minute-based B (`appearances`, 14 leagues, 12 months before and after) needs a baseline of 900+ minutes and at least one league match in the 12 months after `end_date`. `match_measure_possible`: the match-based measure (`player_performances`, seasons) needs at least 10 league matches in the season before the season of the injury (`perf_pre_matches`) and a league row with appearances in the two seasons after the season of the return. Leagues are those of the dictionary in 7.6; cups, youth leagues and lower leagues are not counted.

In [22]:
# 7.11 Which measures are possible for each player (coverage only; no value of either measure is looked at)
LEAGUES = set(TOP5) | SECOND_THIRD | FIRST_VIS | FIRST_NOTVIS              # the leagues of the lists in 7.6; no cups, youth or lower leagues
pl = player_performances[player_performances["competition_id"].isin(LEAGUES)]
pre_season = (W[["inj_id", "player_id", "inj_season"]]
              .merge(pl[["player_id", "season_start", "nb_on_pitch"]], on="player_id")
              .query("season_start == inj_season - 1").groupby("inj_id")["nb_on_pitch"].sum())
W["perf_pre_matches"] = W["inj_id"].map(pre_season).fillna(0)              # league matches in the season before the injury season
post_league = set(yw.loc[yw["competition_id"].isin(LEAGUES), "inj_id"])    # a league row with appearances in the two seasons after the return

W["B_minutes_possible"] = W["l14_post12"] & (W["baseline_group"] == "900+")
W["match_measure_possible"] = (W["perf_pre_matches"] >= 10) & W["inj_id"].isin(post_league)

t = W.groupby("level_after").agg(players=("inj_id", "size"), B_minutes_possible=("B_minutes_possible", "sum"), match_measure_possible=("match_measure_possible", "sum"))
t["neither"] = (~W["B_minutes_possible"] & ~W["match_measure_possible"]).groupby(W["level_after"]).sum()
t.loc["all"] = t.sum()
t

,players,B_minutes_possible,match_measure_possible,neither
level_after,,,,
1 top 5,86,59,67,14
"2 second/third division, top-5 country",42,11,28,14
3 first division elsewhere,25,7,19,6
"5a in squad, no appearances",1,0,0,1
5b no data,4,0,0,4
all,158,77,114,39


### 7.12 The columns

The working table `cohort` gets 45 columns in five groups: A the index injury, B the baseline before it, C what happens after the return, D the level after the return, E flags for the sensitivity analyses. Columns that only served the funnel (`season_name`, `in_range`, `acl_30d`, `top5_pre`, `has24`) are left out: they are constant in W or repeated by another column. Some values are empty on purpose: `first_match` and `days_to_first` for the players without a match in the 24 months after the return, `club_changed` for the players without a match in the 12 months after it. An empty value is not a zero or `False`. Step 8 then keeps 27 of the 45 columns and saves them as the cohort file.

In [23]:
# 7.12 The columns of the working table, in the order of the groups; technical columns of the funnel are left out
GROUPS = {
    "A injury":       ["player_id", "inj_id", "injury_reason", "from_date", "end_date", "days_missed", "games_missed", "absence_band",
                       "inj_season", "ret_season", "summer_start", "pre_start", "post12_end", "post_end"],
    "B baseline":     ["min_top5_pre", "min_pre", "baseline_group", "age_at_injury", "position"],
    "C after return": ["min_post_12", "min_post_24", "l14_post12", "l14_post24", "first_match", "days_to_first", "has12",
                       "club_pre", "club_changed", "transfer_in_window"],
    "D level":        ["level_after", "level_H", "level_M", "level_M_one_source", "level_unstable", "played_top5", "top5_matches"],
    "E flags":        ["other_during", "other_90d_after", "other_injury", "not_first_meniscus", "n_men_before", "few_games_missed",
                       "perf_pre_matches", "B_minutes_possible", "match_measure_possible"],
}
COLUMNS = [c for cols in GROUPS.values() for c in cols]
assert len(COLUMNS) == len(set(COLUMNS)) and set(COLUMNS) <= set(W.columns)

cohort = W[COLUMNS].sort_values("player_id").reset_index(drop=True)
cohort = cohort.astype({"days_missed": int, "min_top5_pre": int, "min_post_12": int, "min_post_24": int,
                        "perf_pre_matches": int, "days_to_first": "Int64"})          # whole numbers; days_to_first stays empty without a match
assert cohort.shape == (158, len(COLUMNS)) and cohort["player_id"].is_unique

print("columns per group:", {g: len(c) for g, c in GROUPS.items()}, "| left out:", [c for c in W.columns if c not in COLUMNS])
empty = cohort.isna().sum()
print("columns with empty values (on purpose):", empty[empty > 0].to_dict())
cohort.dtypes.astype(str).value_counts().rename("columns").to_frame()

columns per group: {'A injury': 14, 'B baseline': 5, 'C after return': 10, 'D level': 7, 'E flags': 9} | left out: ['season_name', 'in_range', 'acl_30d', 'top5_pre', 'has24']
columns with empty values (on purpose): {'first_match': 28, 'days_to_first': 28, 'club_changed': 31}


,columns
bool,14
int64,12
str,6
datetime64[us],6
category,2
int32,2
float64,1
Int64,1
boolean,1


In [24]:
pd.set_option("display.max_columns", 50)
cohort.head(10)

,player_id,inj_id,injury_reason,from_date,end_date,days_missed,games_missed,absence_band,inj_season,ret_season,summer_start,pre_start,post12_end,post_end,min_top5_pre,min_pre,baseline_group,age_at_injury,position,min_post_12,min_post_24,l14_post12,l14_post24,first_match,days_to_first,has12,club_pre,club_changed,transfer_in_window,level_after,level_H,level_M,level_M_one_source,level_unstable,played_top5,top5_matches,other_during,other_90d_after,other_injury,not_first_meniscus,n_men_before,few_games_missed,perf_pre_matches,B_minutes_possible,match_measure_possible
0,2386,513,Meniscus tear,2014-03-06,2014-04-25,51,8,43-60,2013,2013,False,2013-03-06,2015-04-25,2016-04-25,450,450,270-899,34.789870,Goalkeeper,0,0,False,False,NaT,<NA>,False,23,<NA>,False,"2 second/third division, top-5 country","2 second/third division, top-5 country","2 second/third division, top-5 country","2 second/third division, top-5 country",False,False,0,False,False,False,False,0,False,4,False,False
1,4233,1033,Meniscus injury,2017-09-03,2017-12-16,105,16,91+,2017,2017,False,2016-09-03,2018-12-16,2019-12-16,180,180,1-269,32.588638,Goalkeeper,0,0,False,False,NaT,<NA>,False,989,<NA>,False,"2 second/third division, top-5 country","2 second/third division, top-5 country","2 second/third division, top-5 country","2 second/third division, top-5 country",False,False,0,False,False,False,False,0,False,2,False,False
2,5404,1275,Meniscus tear,2016-03-31,2016-05-05,36,8,29-42,2015,2015,False,2015-03-31,2017-05-05,2018-05-05,1743,1743,900+,32.421629,Defender,1074,3453,True,True,2016-05-08,3,True,131,False,False,3 first division elsewhere,1 top 5,3 first division elsewhere,3 first division elsewhere,False,True,15,False,False,False,False,0,False,28,True,True
3,7602,1637,Meniscus injury,2017-10-30,2017-12-10,42,9,29-42,2017,2017,False,2016-10-30,2018-12-10,2019-12-10,2122,2122,900+,35.173169,Midfield,527,527,True,True,2017-12-16,6,True,583,False,False,1 top 5,1 top 5,1 top 5,5 no league match,True,True,11,False,True,True,False,0,False,30,True,False
4,7914,1661,Meniscus tear,2016-03-05,2016-04-11,38,7,29-42,2015,2015,False,2015-03-05,2017-04-11,2018-04-11,3600,3600,900+,32.043806,Goalkeeper,2520,3960,True,True,2016-08-19,130,True,1084,False,False,1 top 5,1 top 5,1 top 5,1 top 5,False,True,35,True,False,True,False,0,False,38,True,True
5,11111,48,Meniscus injury,2017-03-29,2017-07-17,111,11,91+,2016,2017,False,2016-03-29,2018-07-17,2019-07-17,1860,1860,900+,26.067077,Defender,119,2747,True,True,2018-04-14,271,True,29,False,True,1 top 5,1 top 5,1 top 5,1 top 5,False,True,35,True,False,True,False,0,False,28,True,True
6,16520,266,Meniscus injury,2015-03-09,2015-04-27,50,6,43-60,2014,2014,False,2014-03-09,2016-04-27,2017-04-27,2764,2764,900+,29.593429,Defender,2349,2771,True,True,2015-05-02,5,True,24,False,False,1 top 5,1 top 5,1 top 5,1 top 5,False,True,34,False,False,False,False,0,False,29,True,True
7,20003,397,Meniscus injury,2013-09-09,2013-10-15,37,7,29-42,2013,2013,False,2012-09-09,2014-10-15,2015-10-15,386,386,270-899,28.955510,Defender,714,3206,True,True,2013-10-19,4,True,46,True,False,1 top 5,1 top 5,1 top 5,1 top 5,False,True,38,False,True,True,False,0,False,9,False,False
8,21829,455,Meniscus damage,2013-11-20,2014-04-13,145,28,91+,2013,2013,False,2012-11-20,2015-04-13,2016-04-13,360,360,270-899,32.098563,Goalkeeper,1260,1719,True,True,2014-12-02,233,True,458,True,False,1 top 5,1 top 5,1 top 5,1 top 5,False,True,20,False,True,True,False,0,False,5,False,False
9,21887,456,Meniscus injury,2014-12-22,2015-03-06,75,8,61-90,2014,2014,False,2013-12-22,2016-03-06,2017-03-06,2225,2225,900+,30.179329,Defender,569,852,True,True,2015-04-11,36,True,252,False,False,"2 second/third division, top-5 country",1 top 5,"2 second/third division, top-5 country","2 second/third division, top-5 country",False,True,14,False,True,True,False,0,False,27,True,True


## Step 8: The final cohort table

The working table `cohort` (step 7.12) has 45 columns. They were added group by group, as each part of the plan was built. Before saving the cohort we checked which of them are really needed. 27 stay. The other 18 are left out for one of four reasons:

| Reason | Columns |
|---|---|
| Recomputable from the kept columns | `inj_season`, `ret_season`, `pre_start`, `post12_end`, `post_end` (from the dates); `l14_post12`, `l14_post24` (from `min_post_12` and `min_post_24`); `first_match` (`end_date` + `days_to_first`); `has12` (`club_changed` is not empty); `level_M` (`level_after` before level 5 is split into 5a and 5b) |
| A key, a helper or a check only | `inj_id` (one injury per player, `player_id` is unique); `club_pre` (helper of `club_changed`); `transfer_in_window` (a check of `club_changed` against the `transfers` table) |
| The flag is enough | `n_men_before` (the count of earlier meniscal injuries; `not_first_meniscus` is kept) |
| Dropped by decision | `few_games_missed` (all 5 players are already in `summer_start`), `level_M_one_source`, `level_H`, `level_unstable` (described in steps 7.8 and 7.9 and in the audit) |

### 8.1 The columns we keep

In [25]:
# 8.1 The columns we keep: 27 of the 45
FINAL_GROUPS = {
    "A injury":       ["player_id", "injury_reason", "from_date", "end_date", "days_missed", "games_missed", "absence_band", "summer_start"],
    "B baseline":     ["min_top5_pre", "min_pre", "baseline_group", "age_at_injury", "position"],
    "C after return": ["min_post_12", "min_post_24", "days_to_first", "club_changed"],
    "D level":        ["level_after", "top5_matches", "played_top5"],
    "E flags":        ["other_during", "other_90d_after", "other_injury", "not_first_meniscus",
                       "perf_pre_matches", "B_minutes_possible", "match_measure_possible"],
}
FINAL = [c for cols in FINAL_GROUPS.values() for c in cols]
assert len(FINAL) == len(set(FINAL)) and set(FINAL) <= set(cohort.columns)

dropped = [c for c in cohort.columns if c not in FINAL]
cohort_final = cohort[FINAL].copy()
assert cohort_final.shape == (158, len(FINAL)) and cohort_final["player_id"].is_unique

print("columns per group:", {g: len(c) for g, c in FINAL_GROUPS.items()}, "| kept:", len(FINAL), "| dropped:", len(dropped))
print("dropped:", dropped)
empty = cohort_final.isna().sum()
print("columns with empty values (on purpose):", empty[empty > 0].to_dict()) 
cohort_final.head(3)

columns per group: {'A injury': 8, 'B baseline': 5, 'C after return': 4, 'D level': 3, 'E flags': 7} | kept: 27 | dropped: 18
dropped: ['inj_id', 'inj_season', 'ret_season', 'pre_start', 'post12_end', 'post_end', 'l14_post12', 'l14_post24', 'first_match', 'has12', 'club_pre', 'transfer_in_window', 'level_H', 'level_M', 'level_M_one_source', 'level_unstable', 'n_men_before', 'few_games_missed']
columns with empty values (on purpose): {'days_to_first': 28, 'club_changed': 31}


,player_id,injury_reason,from_date,end_date,days_missed,games_missed,absence_band,summer_start,min_top5_pre,min_pre,baseline_group,age_at_injury,position,min_post_12,min_post_24,days_to_first,club_changed,level_after,top5_matches,played_top5,other_during,other_90d_after,other_injury,not_first_meniscus,perf_pre_matches,B_minutes_possible,match_measure_possible
0,2386,Meniscus tear,2014-03-06,2014-04-25,51,8,43-60,False,450,450,270-899,34.789870,Goalkeeper,0,0,<NA>,<NA>,"2 second/third division, top-5 country",0,False,False,False,False,False,4,False,False
1,4233,Meniscus injury,2017-09-03,2017-12-16,105,16,91+,False,180,180,1-269,32.588638,Goalkeeper,0,0,<NA>,<NA>,"2 second/third division, top-5 country",0,False,False,False,False,False,2,False,False
2,5404,Meniscus tear,2016-03-31,2016-05-05,36,8,29-42,False,1743,1743,900+,32.421629,Defender,1074,3453,3,False,3 first division elsewhere,15,True,False,False,False,False,28,True,True


### 8.2 Evidence for the columns we leave out

Each check below is a rule that has to hold for all 158 players. The first ten rows show that the column can be recomputed from a column we keep. The last three rows are the facts behind the decisions to drop `few_games_missed`, `level_unstable` and `level_H`: the flag `few_games_missed` adds nothing next to `summer_start`, and the top-5 part of `level_H` is exactly `played_top5`. The assertion at the end stops the notebook if any rule fails.

In [26]:
# 8.2 Evidence: the 18 columns we leave out can be recomputed, or are not needed
co = cohort                                                      # short name for the 45-column table
has_match = co["first_match"].notna()
first_back = co["end_date"] + pd.to_timedelta(co["days_to_first"].astype("float"), unit="D")
season = lambda d: d.dt.year - (d.dt.month < 7).astype(int)     # July-June seasons, named by the year in which the season starts

rows = [
    ("inj_id", "one injury per player: both ids are unique",
     co["inj_id"].is_unique and co["player_id"].is_unique),
    ("inj_season, ret_season", "season of from_date and of end_date",
     (co["inj_season"] == season(co["from_date"])).all() and (co["ret_season"] == season(co["end_date"])).all()),
    ("pre_start", "from_date - 12 months",
     (co["pre_start"] == co["from_date"] - pd.DateOffset(months=12)).all()),
    ("post12_end", "end_date + 12 months",
     (co["post12_end"] == co["end_date"] + pd.DateOffset(months=12)).all()),
    ("post_end", "end_date + 24 months",
     (co["post_end"] == co["end_date"] + pd.DateOffset(months=24)).all()),
    ("l14_post12, l14_post24", "min_post_12 > 0 and min_post_24 > 0",
     (co["l14_post12"] == (co["min_post_12"] > 0)).all() and (co["l14_post24"] == (co["min_post_24"] > 0)).all()),
    ("first_match", "end_date + days_to_first (empty in the same 28 rows)",
     first_back[has_match].equals(co["first_match"][has_match]) and (co["days_to_first"].notna() == has_match).all()),
    ("has12", "club_changed is not empty",
     (co["has12"] == co["club_changed"].notna()).all()),
    ("level_M", "level_after, except that level 5 is split into 5a and 5b",
     (co["level_after"].str[0] == co["level_M"].str[0]).all()
     and co.loc[co["level_M"] != co["level_after"], "level_M"].eq("5 no league match").all()),
    ("n_men_before", "not_first_meniscus = n_men_before > 0",
     (co["not_first_meniscus"] == (co["n_men_before"] > 0)).all()),
    ("few_games_missed", "games_missed <= 2, and all these players are in summer_start",
     (co["few_games_missed"] == (co["games_missed"] <= 2)).all() and co.loc[co["few_games_missed"], "summer_start"].all()),
    ("level_unstable", "level_M_one_source differs from level_M",
     (co["level_unstable"] == (co["level_M_one_source"] != co["level_M"])).all()),
    ("level_H", "top 5 in level_H exactly when played_top5",
     ((co["level_H"].str[0] == "1") == co["played_top5"]).all()),
]
evidence = pd.DataFrame(rows, columns=["left out", "what it equals", "holds"])
with pd.option_context("display.max_colwidth", None):            # show the full text of the second column
    display(evidence)
assert evidence["holds"].all()

print("few_games_missed:", int(co["few_games_missed"].sum()), "players, all in summer_start (", int(co["summer_start"].sum()), "players )")
print("level_H differs from level_M for", int((co["level_H"] != co["level_M"]).sum()), "players; of them",
      int((co["played_top5"] & (co["level_M"].str[0] != "1")).sum()), "played in the top 5 but have most matches below it")
print("level_unstable:", int(co["level_unstable"].sum()), "players")
print("not covered by a check: club_pre (helper of club_changed) and transfer_in_window (a check against the transfers table)")

,left out,what it equals,holds
0,inj_id,one injury per player: both ids are unique,True
1,"inj_season, ret_season",season of from_date and of end_date,True
2,pre_start,from_date - 12 months,True
3,post12_end,end_date + 12 months,True
4,post_end,end_date + 24 months,True
5,"l14_post12, l14_post24",min_post_12 > 0 and min_post_24 > 0,True
6,first_match,end_date + days_to_first (empty in the same 28 rows),True
7,has12,club_changed is not empty,True
8,level_M,"level_after, except that level 5 is split into 5a and 5b",True
9,n_men_before,not_first_meniscus = n_men_before > 0,True


few_games_missed: 5 players, all in summer_start ( 20 players )
level_H differs from level_M for 31 players; of them 30 played in the top 5 but have most matches below it
level_unstable: 11 players
not covered by a check: club_pre (helper of club_changed) and transfer_in_window (a check against the transfers table)


### 8.3 Save the final cohort

The file `data/processed/cohort.csv` is the only thing the analysis notebook reads: one row per player, 27 columns. It is a plain CSV file, so it can be opened anywhere. A CSV file does not store column types: the two band columns come back as text, `days_to_first` as decimal numbers, `club_changed` as an `object` column, and the dates as text unless we ask for them with `parse_dates`. Step 8.4 therefore reads the file back with `parse_dates` and a fixed type for every other column that needs one, and checks that the result equals `cohort_final`; the analysis notebook uses the same recipe.

In [27]:
# 8.3 Save the final cohort: one row per player, as a CSV file in data/processed
from pathlib import Path

COHORT_FILE = "cohort.csv"
DATA_DIR = next((p for p in [Path("data"), Path("../data")] if p.exists()), Path("data"))    # the same places as the raw tables
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

cohort_path = OUT_DIR / COHORT_FILE
cohort_final.to_csv(cohort_path, index=False)                    # index=False: the row numbers are not data

lines = cohort_path.read_text().splitlines()
assert len(lines) == len(cohort_final) + 1                       # the header plus one line per player
print("saved:", cohort_path.resolve())
print("rows:", len(cohort_final), "| columns:", cohort_final.shape[1], "| size:", f"{cohort_path.stat().st_size / 1024:.1f} KB")
print(lines[0])                                                  # the header
print(*lines[1:3], sep="\n")                                     # the first two players

saved: /workspaces/meniscus-injuries-football/data/processed/cohort.csv
rows: 158 | columns: 27 | size: 28.6 KB
player_id,injury_reason,from_date,end_date,days_missed,games_missed,absence_band,summer_start,min_top5_pre,min_pre,baseline_group,age_at_injury,position,min_post_12,min_post_24,days_to_first,club_changed,level_after,top5_matches,played_top5,other_during,other_90d_after,other_injury,not_first_meniscus,perf_pre_matches,B_minutes_possible,match_measure_possible
2386,Meniscus tear,2014-03-06,2014-04-25,51,8,43-60,False,450,450,270-899,34.789869952087614,Goalkeeper,0,0,,,"2 second/third division, top-5 country",0,False,False,False,False,False,4,False,False
4233,Meniscus injury,2017-09-03,2017-12-16,105,16,91+,False,180,180,1-269,32.5886379192334,Goalkeeper,0,0,,,"2 second/third division, top-5 country",0,False,False,False,False,False,2,False,False


### 8.4 Read the file back and compare

We read `cohort.csv` back with a fixed type for each column that a CSV file does not remember (the two band columns, `days_to_first` and `club_changed`) and compare the result with `cohort_final`: same columns, order, types and values. If they differ, the cell stops with an error. `COHORT_DTYPES` and `read_cohort` are the recipe for reading the cohort in the analysis notebook.

In [28]:
# 8.4 Read the file back with a fixed type for every column and compare it with cohort_final
COHORT_DTYPES = {                                                # what a CSV file does not remember
    "absence_band":   pd.CategoricalDtype(["29-42", "43-60", "61-90", "91+"], ordered=True),
    "baseline_group": pd.CategoricalDtype(["1-269", "270-899", "900+"], ordered=True),
    "days_to_first":  "Int64",                                   # whole days, empty when there is no match
    "club_changed":   "boolean",                                 # True / False, empty when there is no match in the 12 months
}

def read_cohort(path):
    return pd.read_csv(path, parse_dates=["from_date", "end_date"], dtype=COHORT_DTYPES, float_precision="round_trip")

cohort_back = read_cohort(cohort_path)
pd.testing.assert_frame_equal(cohort_back, cohort_final)         # same columns, order, types and values
print("the file read back equals cohort_final:", cohort_back.shape)

# what happens without the types (for the record)
plain = pd.read_csv(cohort_path, parse_dates=["from_date", "end_date"])
different = {c: (str(plain[c].dtype), str(cohort_final[c].dtype)) for c in plain.columns if plain[c].dtype != cohort_final[c].dtype}
print("columns that change type without COHORT_DTYPES (read as, should be):", different)

the file read back equals cohort_final: (158, 27)
columns that change type without COHORT_DTYPES (read as, should be): {'absence_band': ('str', 'category'), 'baseline_group': ('str', 'category'), 'days_to_first': ('float64', 'Int64'), 'club_changed': ('object', 'boolean')}
